<a href="https://colab.research.google.com/github/satrishabh/ML-DL-GENAI/blob/main/LangChain_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [110]:
!pip install -q langchain langchain-google-genai google-generativeai
!pip install langchain-core langchain-community duckduckgo-search langchain_experimental

In [111]:
!pip install pypdf

In [112]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai.llms import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [113]:
os.environ["GOOGLE_API_KEY"]=""
llm=ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [114]:
loader=PyPDFLoader("/content/My_AxiOraa_Ltd_Synthetic_Annual_Report_2025.pdf")
documents=loader.load()

report=""
for page in documents:
  report+=page.page_content+'\n'
print("total pages",len(documents))
print(report[:1000])

total pages 9
AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable 

In [115]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [116]:
embedding=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)# embedding model we have

In [117]:
from langchain_community.vectorstores import FAISS

In [118]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [119]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks=text_splitter.split_documents(documents)

In [120]:
!pip install faiss-cpu
vector_store=FAISS.from_documents(documents=chunks,embedding=embedding)

In [121]:
query="what investments did Axionrra make in AI"

In [122]:
result=vector_store.similarity_search_with_score(query, k=5)
for i, (doc,score) in enumerate(result, start=1):
  print(f"result{i}")
  print(f"similarity:{score}")

result1
similarity:0.5704220533370972
result2
similarity:0.5855493545532227
result3
similarity:0.6042476892471313
result4
similarity:0.6251094341278076
result5
similarity:0.6335211992263794


In [123]:
rag_prompt=ChatPromptTemplate.from_template(
    """
      You are a senior consultant and your job is to answer the users's questions only by using the retrieve context.
      if the answer cannot be found in the context, simply say - "i couldn' find theis infomation"

      Context:{context}
      Question:{question}

      provide:
        1.Answer
        2.Supporting evidence
    """
)

In [124]:
rag_chain=(
    rag_prompt | llm | StrOutputParser()
)

In [125]:
question="Summarize the AI Investment made by axioraa"

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)
retrieved_docs=retriever.invoke(question)
context="\n\n".join(
      [doc.page_content for doc in retrieved_docs]
    )
response=rag_chain.invoke(
    {
        "context":context,
        "question":question
    }

)
print(response)

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 28.993081021s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '